In [16]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.ticker import MaxNLocator

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

plt.rcParams.update({
    "font.family": "serif",
    "font.size": 10,
    "axes.labelsize": 11,
    "axes.titlesize": 11,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "figure.dpi": 150,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.grid": True,
    "grid.alpha": 0.25,
    "grid.linewidth": 0.6,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "legend.framealpha": 0.9,
    "legend.edgecolor": "0.8",
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
})

BLUE   = "#2563EB"
ORANGE = "#EA580C"
GREEN  = "#16A34A"


In [17]:
def _build_inverse_theta(angles, sx, sy):
    thetas = []
    for i in range(angles.shape[0]):
        a   = angles[i].item()
        ix  = 1.0 / max(sx[i].item(), 1e-6)
        iy  = 1.0 / max(sy[i].item(), 1e-6)
        c, s = np.cos(a), np.sin(a)
        thetas.append(torch.tensor(
            [[ c * ix,  s * iy, 0.0],
             [-s * ix,  c * iy, 0.0]], dtype=torch.float32))
    return torch.stack(thetas).to(device)


def apply_transform(x, angles, sx, sy, mode="bilinear"):
    theta = _build_inverse_theta(angles, sx, sy)
    grid  = F.affine_grid(theta, x.size(), align_corners=False)
    return F.grid_sample(x, grid, mode=mode, padding_mode="zeros", align_corners=False)


def sample_transform(batch_size, p_identity=0.5, use_rotation=True, use_scaling=True):
    p      = torch.rand(batch_size)
    angles = torch.zeros(batch_size)
    sx     = torch.ones(batch_size)
    sy     = torch.ones(batch_size)
    apply  = p >= p_identity
    for i in range(batch_size):
        if apply[i]:
            if use_rotation:
                angles[i] = float(np.random.choice([np.pi/2, np.pi, 3*np.pi/2]))
            if use_scaling:
                sx[i] = float(np.random.uniform(0.25, 1.0))
                sy[i] = float(np.random.uniform(0.25, 1.0))
    return angles.to(device), sx.to(device), sy.to(device), apply.to(device)


In [18]:
class Encoder(nn.Module):
    def __init__(self, in_channels=3, latent_ch=4, base_ch=32):
        super().__init__()
        self.body = nn.Sequential(
            nn.Conv2d(in_channels, base_ch,     3, stride=2, padding=1),
            nn.SiLU(),
            nn.Conv2d(base_ch,     base_ch * 2, 3, stride=2, padding=1),
            nn.SiLU(),
            nn.Conv2d(base_ch * 2, base_ch * 2, 3, stride=1, padding=1),
            nn.SiLU(),
        )
        self.conv_mu     = nn.Conv2d(base_ch * 2, latent_ch, 1)
        self.conv_logvar = nn.Conv2d(base_ch * 2, latent_ch, 1)

    def forward(self, x):
        h = self.body(x)
        return self.conv_mu(h), self.conv_logvar(h)


In [19]:
class Decoder(nn.Module):
    def __init__(self, latent_ch=4, out_channels=3, base_ch=32):
        super().__init__()
        self.body = nn.Sequential(
            nn.Conv2d(latent_ch,   base_ch * 2, 3, stride=1, padding=1),
            nn.SiLU(),
            nn.ConvTranspose2d(base_ch * 2, base_ch * 2, 4, stride=2, padding=1),
            nn.SiLU(),
            nn.ConvTranspose2d(base_ch * 2, base_ch,     4, stride=2, padding=1),
            nn.SiLU(),
            nn.Conv2d(base_ch, out_channels, 3, stride=1, padding=1),
            nn.Sigmoid(),
        )

    def forward(self, z):
        return self.body(z)


In [20]:
class EQVAE(nn.Module):
    def __init__(self, latent_ch=4, base_ch=32):
        super().__init__()
        self.encoder = Encoder(latent_ch=latent_ch, base_ch=base_ch)
        self.decoder = Decoder(latent_ch=latent_ch, base_ch=base_ch)

    def reparameterise(self, mu, logvar):
        if self.training:
            return mu + torch.exp(0.5 * logvar) * torch.randn_like(mu)
        return mu

    def encode(self, x):
        mu, logvar = self.encoder(x)
        return self.reparameterise(mu, logvar), mu, logvar

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        z, mu, logvar = self.encode(x)
        return self.decode(z), mu, logvar


In [21]:
def kl_loss(mu, logvar):
    return -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())


def reconstruction_loss(x_hat, x_target):
    return F.mse_loss(x_hat, x_target)


def eqvae_loss(model, x, p_identity=0.5, lambda_kl=1e-4,
               use_rotation=True, use_scaling=True):
    B = x.size(0)
    angles, sx, sy, apply_mask = sample_transform(B, p_identity, use_rotation, use_scaling)
    z, mu, logvar = model.encode(x)
    z_transformed = apply_transform(z, angles, sx, sy, mode="bilinear")
    with torch.no_grad():
        x_transformed = apply_transform(x, angles, sx, sy, mode="bicubic")
    mask     = apply_mask.float().view(B, 1, 1, 1)
    z_final  = mask * z_transformed + (1 - mask) * z
    x_target = mask * x_transformed + (1 - mask) * x
    x_hat = model.decode(z_final)
    rec   = reconstruction_loss(x_hat, x_target)
    kl    = kl_loss(mu, logvar)
    return rec + lambda_kl * kl, rec, kl, x_hat, x_target


In [22]:
def get_cifar10(batch_size=64, train=True, data_root="../data"):
    tf = transforms.Compose([transforms.ToTensor()])
    ds = torchvision.datasets.CIFAR10(root=data_root, train=train, download=True, transform=tf)
    return DataLoader(ds, batch_size=batch_size, shuffle=train,
                      num_workers=2, pin_memory=(device.type == "cuda"))

train_loader = get_cifar10(batch_size=64, train=True)
val_loader   = get_cifar10(batch_size=64, train=False)


100%|██████████| 170M/170M [00:25<00:00, 6.77MB/s] 


In [ ]:
model     = EQVAE(latent_ch=4, base_ch=32).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)


In [ ]:
def train_epoch(model, loader, optimizer, p_identity=0.5, lambda_kl=1e-4):
    model.train()
    total_l = rec_l = kl_l = 0.0
    for x, _ in loader:
        x = x.to(device)
        optimizer.zero_grad()
        loss, rec, kl, _, _ = eqvae_loss(model, x, p_identity, lambda_kl)
        loss.backward()
        optimizer.step()
        total_l += loss.item()
        rec_l   += rec.item()
        kl_l    += kl.item()
    n = len(loader)
    return total_l / n, rec_l / n, kl_l / n


NUM_EPOCHS = 5
history = {"loss": [], "rec": [], "kl": []}

for epoch in range(1, NUM_EPOCHS + 1):
    loss, rec, kl = train_epoch(model, train_loader, optimizer)
    history["loss"].append(loss)
    history["rec"].append(rec)
    history["kl"].append(kl)
    print(f"Epoch {epoch}/{NUM_EPOCHS}  loss={loss:.4f}  rec={rec:.4f}  kl={kl:.6f}")


In [ ]:
epochs = list(range(1, len(history["loss"]) + 1))

fig, axes = plt.subplots(1, 2, figsize=(8, 3))

ax = axes[0]
ax.plot(epochs, history["rec"], color=BLUE, lw=2, marker="o", ms=5, zorder=3)
ax.fill_between(epochs, history["rec"], alpha=0.10, color=BLUE)
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE")
ax.set_title("Reconstruction Loss")
ax.xaxis.set_major_locator(MaxNLocator(integer=True))

ax = axes[1]
ax.plot(epochs, history["kl"], color=ORANGE, lw=2, marker="s", ms=5, zorder=3)
ax.fill_between(epochs, history["kl"], alpha=0.10, color=ORANGE)
ax.set_xlabel("Epoch")
ax.set_ylabel("Nats")
ax.set_title("KL Divergence")
ax.xaxis.set_major_locator(MaxNLocator(integer=True))

fig.suptitle("EQ-VAE Training Dynamics", fontsize=12, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
CIFAR_CLASSES = ["airplane", "automobile", "bird", "cat", "deer",
                 "dog", "frog", "horse", "ship", "truck"]

model.eval()
x_batch, batch_labels = next(iter(val_loader))
x_batch = x_batch[:8].to(device)

with torch.no_grad():
    recon, _, _ = model(x_batch)

n = x_batch.size(0)
fig, axes = plt.subplots(2, n, figsize=(n * 1.5, 3.4),
                         gridspec_kw={"hspace": 0.04, "wspace": 0.04})

row_labels = ["Input", "Recon."]
for r, (imgs, rl) in enumerate([(x_batch, row_labels[0]), (recon, row_labels[1])]):
    for c in range(n):
        ax = axes[r, c]
        ax.imshow(imgs[c].cpu().permute(1, 2, 0).clamp(0, 1).numpy(),
                  interpolation="lanczos")
        ax.set_xticks([])
        ax.set_yticks([])
        for sp in ax.spines.values():
            sp.set_visible(False)
        if c == 0:
            ax.set_ylabel(rl, fontsize=10, fontweight="bold",
                          rotation=0, labelpad=36, va="center")
    for c in range(n):
        axes[0, c].set_title(CIFAR_CLASSES[batch_labels[c].item()],
                             fontsize=8, pad=3)

fig.suptitle("EQ-VAE Reconstruction Quality", fontsize=12,
             fontweight="bold", y=1.01)
plt.show()


In [ ]:
model.eval()

all_z, all_x = [], []
with torch.no_grad():
    for x, _ in val_loader:
        z, _, _ = model.encode(x.to(device))
        all_z.append(z.cpu())
        all_x.append(x)
        if sum(t.shape[0] for t in all_z) >= 512:
            break

z_stack = torch.cat(all_z)
x_stack = torch.cat(all_x)

B, C, H, W = z_stack.shape
z_flat = z_stack.permute(0, 2, 3, 1).reshape(-1, C).float()
z_cent = z_flat - z_flat.mean(0, keepdim=True)
_, _, Vh = torch.linalg.svd(z_cent, full_matrices=False)
z_pca = (z_cent @ Vh[:3].T).reshape(B, H, W, 3).permute(0, 3, 1, 2)
lo, hi = z_pca.min(), z_pca.max()
z_pca_norm = (z_pca - lo) / (hi - lo + 1e-6)

n_show = 8
fig, axes = plt.subplots(2, n_show, figsize=(n_show * 1.5, 3.4),
                         gridspec_kw={"hspace": 0.04, "wspace": 0.04})

row_labels = ["Image", "Latent\nPC 1-3"]
for r, (imgs, rl) in enumerate([(x_stack, row_labels[0]), (z_pca_norm, row_labels[1])]):
    for c in range(n_show):
        ax = axes[r, c]
        ax.imshow(imgs[c].permute(1, 2, 0).clamp(0, 1).numpy(),
                  interpolation="lanczos")
        ax.set_xticks([])
        ax.set_yticks([])
        for sp in ax.spines.values():
            sp.set_visible(False)
        if c == 0:
            ax.set_ylabel(rl, fontsize=10, fontweight="bold",
                          rotation=0, labelpad=42, va="center")

fig.suptitle("First 3 Principal Components of the EQ-VAE Latent Space",
             fontsize=12, fontweight="bold", y=1.01)
plt.show()


In [ ]:
model.eval()

x_vis, _ = next(iter(val_loader))
x_vis = x_vis[:5].to(device)
B     = x_vis.size(0)

a_fix = torch.zeros(B, device=device)
s_fix = torch.full((B,), 0.5, device=device)

with torch.no_grad():
    x_t       = apply_transform(x_vis, a_fix, s_fix, s_fix, mode="bicubic")
    z, _, _   = model.encode(x_vis)
    recon_x   = model.decode(z)
    z_xt, _,_ = model.encode(x_t)
    recon_a   = model.decode(z_xt)
    z_t       = apply_transform(z, a_fix, s_fix, s_fix, mode="bilinear")
    recon_b   = model.decode(z_t)

rows = [
    (x_vis,   r"$x$"),
    (x_t,     r"$\tau \circ x$"),
    (recon_x, r"$D(E(x))$"),
    (recon_a, r"$D(E(\tau \!\circ\! x))$"),
    (recon_b, r"$D(\tau \!\circ\! E(x))$"),
]

n_rows, n_cols = len(rows), B
fig, axes = plt.subplots(n_rows, n_cols,
                         figsize=(n_cols * 1.4, n_rows * 1.4),
                         gridspec_kw={"hspace": 0.06, "wspace": 0.04})

for r, (tensor, label) in enumerate(rows):
    for c in range(n_cols):
        ax = axes[r, c]
        img = tensor[c].cpu().permute(1, 2, 0).clamp(0, 1).numpy()
        ax.imshow(img, interpolation="lanczos")
        ax.set_xticks([])
        ax.set_yticks([])
        for sp in ax.spines.values():
            sp.set_visible(False)
    axes[r, 0].set_ylabel(label, fontsize=10, rotation=0, labelpad=64, va="center")

for c in range(n_cols):
    axes[0, c].set_title(f"Sample {c + 1}", fontsize=8, pad=3)

fig.suptitle(
    r"Equivariance: $D(\tau \circ E(x)) \approx D(E(\tau \circ x)) \approx \tau \circ x$",
    fontsize=11, fontweight="bold", y=1.01)
plt.show()


In [ ]:
@torch.no_grad()
def equivariance_error(model, loader, scale, n_batches=20):
    model.eval()
    errs = []
    for i, (x, _) in enumerate(loader):
        if i >= n_batches:
            break
        x  = x.to(device)
        B  = x.size(0)
        a  = torch.zeros(B, device=device)
        sx = torch.full((B,), scale, device=device)
        x_t     = apply_transform(x, a, sx, sx, mode="bicubic")
        z, _, _ = model.encode(x)
        z_t     = apply_transform(z, a, sx, sx, mode="bilinear")
        recon_b = model.decode(z_t)
        errs.append(F.mse_loss(recon_b, x_t).item())
    return float(np.mean(errs))


scale_labels = ["0.25", "0.50", "0.75"]
scale_vals   = [0.25, 0.50, 0.75]
errors       = [equivariance_error(model, val_loader, s) for s in scale_vals]

fig, ax = plt.subplots(figsize=(4.5, 3.4))
x_pos = np.arange(len(scale_labels))
bars  = ax.bar(x_pos, errors, width=0.5, color=BLUE, alpha=0.85,
               edgecolor="white", linewidth=0.8, zorder=3)

for bar, val in zip(bars, errors):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + max(errors) * 0.015,
            f"{val:.4f}", ha="center", va="bottom", fontsize=9)

ax.set_xticks(x_pos)
ax.set_xticklabels([f"$s = {l}$" for l in scale_labels])
ax.set_xlabel("Scaling factor $s$")
ax.set_ylabel(r"MSE$(\tau \!\circ\! x,\ D(\tau \!\circ\! E(x)))$")
ax.set_title("Equivariance Error vs. Scale", fontweight="bold")
ax.set_ylim(0, max(errors) * 1.30)
ax.yaxis.set_major_locator(MaxNLocator(4))

plt.tight_layout()
plt.show()
